In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import copy
import random
import os
from tqdm import tqdm

# ==========================================
# 1. UTILS
# ==========================================
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

def get_device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_class_subset(dataset, classes):
    indices = [i for i, label in enumerate(dataset.targets) if label in classes]
    return Subset(dataset, indices)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_replay_buffer(dataset, classes, per_class=200):
    # Reduced buffer size, just enough for anchoring
    indices = []
    counts = {k:0 for k in classes}
    for i, label in enumerate(dataset.targets):
        if label in classes and counts[label] < per_class:
            indices.append(i)
            counts[label] += 1
    return Subset(dataset, indices)

# ==========================================
# 2. ARCHITECTURE
# ==========================================
class CosineLinear(nn.Module):
    def __init__(self, in_features, out_features, sigma=10.0):
        super(CosineLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.sigma = nn.Parameter(torch.Tensor([sigma]))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

    def forward(self, input):
        return self.sigma * F.linear(F.normalize(input, p=2, dim=1), F.normalize(self.weight, p=2, dim=1))

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False), nn.BatchNorm2d(out_c))
    def forward(self, x): return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.final_conv = nn.Conv2d(256, 64, 1)
        
    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = []
        layers.append(ResidualBlock(in_c, out_c, stride))
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = self.final_conv(x)
        return x

class StructuralFusionModule(nn.Module):
    def __init__(self, in_channels, old_dim, new_dim, novelty_score):
        super().__init__()
        self.novelty_score = novelty_score
        self.has_expansion = new_dim > 0
        
        self.proj_reuse = nn.Sequential(
            nn.Conv2d(in_channels, old_dim, 1, bias=False),
            nn.BatchNorm2d(old_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        self.gate_reuse = nn.Parameter(torch.tensor([0.0])) 
        
        if self.has_expansion:
            self.proj_expand = nn.Sequential(
                nn.Conv2d(in_channels, new_dim, 1, bias=False),
                nn.BatchNorm2d(new_dim),
                nn.ReLU()
            )

    def forward(self, x, old_memory_detached):
        reuse_scale = max(0.1, 1.0 - self.novelty_score)
        delta_old = self.proj_reuse(x) * torch.sigmoid(self.gate_reuse) * reuse_scale
        
        feat_new = None
        if self.has_expansion:
            raw_new = self.proj_expand(x)
            # Center features relative to old memory mean (Soft Orthogonality)
            feat_new = raw_new - raw_new.mean(dim=(2,3), keepdim=True)
            
        return delta_old, feat_new

class GlobalModel(nn.Module):
    def __init__(self, n_specs, ch, n_classes, old_dim=64, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.old_n_specs = 1 
        self.old_proj = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(self.old_n_specs)])
        self.old_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch, ch//4, 1), nn.ReLU(), 
            nn.Conv2d(ch//4, ch, 1), nn.Sigmoid()
        )
        self.old_bottleneck = nn.Sequential(nn.Conv2d(ch, old_dim, 1), nn.ReLU())
        
        self.old_dim = old_dim
        self.new_dim = new_dim
        if new_dim > 0:
            self.fusion = StructuralFusionModule(ch, old_dim, new_dim, novelty_score)
        else:
            self.fusion = None
            
        self.pool = nn.AdaptiveAvgPool2d(1)
        # Cosine Classifier
        self.classifier = CosineLinear(old_dim + new_dim, n_classes)

    def forward_old(self, feats):
        f = feats[0] 
        p = self.old_proj[0](f)
        w = self.old_gate(p)
        z = p * w
        return self.old_bottleneck(z)

    def forward(self, feats):
        # 1. Old Memory
        old_memory = self.forward_old(feats)
        
        # 2. Fusion
        if self.fusion is not None:
            spec_new = feats[-1]
            delta_old, feat_new = self.fusion(spec_new, old_memory.detach())
            enhanced_old = old_memory + delta_old
            if feat_new is not None:
                final_features_new_path = torch.cat([enhanced_old, feat_new], dim=1)
            else:
                final_features_new_path = enhanced_old
        else:
            final_features_new_path = old_memory

        # 3. Unified Classification (Cosine)
        
        # We assume the classifier handles the splitting/normalization internally via CosineLinear
        # But for structural isolation, we must manually slice the inputs for the 'old' classes
        # to ensure they ONLY see the old dimensions.
        
        # This is tricky with a single CosineLinear layer. 
        # Standard CosineLinear normalizes the FULL input vector. 
        # If we pass [Old, New], it normalizes the whole thing.
        # But Old classes should ignore New.
        
        # Solution: Construct logits manually to enforce mask.
        
        # Normalize weights
        W_norm = F.normalize(self.classifier.weight, p=2, dim=1)
        
        # Path A: Old Classes (0-4)
        # Input: Pure Old Memory
        flat_old = self.pool(old_memory).flatten(1)
        norm_in_old = F.normalize(flat_old, p=2, dim=1)
        # Weight Slices: [0:5, 0:old_dim]
        # We must re-normalize the slice to be mathematically correct cosine
        W_old_slice = self.classifier.weight[:5, :self.old_dim]
        W_old_norm = F.normalize(W_old_slice, p=2, dim=1)
        logits_old = F.linear(norm_in_old, W_old_norm) * self.classifier.sigma
        
        # Path B: New Classes (5-9)
        # Input: Enhanced Full Memory
        flat_new = self.pool(final_features_new_path).flatten(1)
        norm_in_new = F.normalize(flat_new, p=2, dim=1)
        # Weight Slices: [5:, :]
        W_new_slice = self.classifier.weight[5:, :]
        W_new_norm = F.normalize(W_new_slice, p=2, dim=1)
        logits_new = F.linear(norm_in_new, W_new_norm) * self.classifier.sigma
        
        return torch.cat([logits_old, logits_new], dim=1), final_features_new_path

class KFN(nn.Module):
    def __init__(self, n_classes=5, n_specs=1, old_dim=64, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        self.global_model = GlobalModel(n_specs, 64, n_classes, old_dim, new_dim, novelty_score)
    def forward(self, x): 
        return self.global_model([s(x) for s in self.specialists])

# ==========================================
# 3. EXPANSION & TRAINING
# ==========================================

def expand_kfn(old_model, trained_spec, new_dim, novelty_score):
    new_kfn = KFN(n_classes=10, n_specs=2, old_dim=64, new_dim=new_dim, novelty_score=novelty_score).to(get_device())
    
    new_kfn.specialists[0].load_state_dict(old_model.specialists[0].state_dict())
    new_kfn.specialists[1].load_state_dict(trained_spec.state_dict())
    
    old_g = old_model.global_model
    new_g = new_kfn.global_model
    
    new_g.old_proj.load_state_dict(old_g.old_proj.state_dict())
    new_g.old_gate.load_state_dict(old_g.old_gate.state_dict())
    new_g.old_bottleneck.load_state_dict(old_g.old_bottleneck.state_dict())
    
    with torch.no_grad():
        new_g.classifier.weight[:5, :64] = old_g.classifier.weight
        new_g.classifier.sigma.data = old_g.classifier.sigma.data.clone()
        nn.init.kaiming_normal_(new_g.classifier.weight[5:, :], nonlinearity='relu')
        new_g.classifier.weight[:5, 64:].zero_()
        
    return new_kfn

def compute_novelty(model, specialist, loader, device):
    print("\n[Phase 3A] Novelty Analysis...")
    model.eval()
    specialist.eval()
    scores = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc="Scanning", leave=False):
            x = x.to(device)
            f_new = F.normalize(specialist(x).flatten(1), p=2, dim=1)
            spec_feat = model.specialists[0](x)
            f_old_map = model.global_model.forward_old([spec_feat])
            f_old = F.normalize(f_old_map.flatten(1), p=2, dim=1)
            proj = torch.sum(f_new * f_old, dim=1, keepdim=True) * f_old
            residual = f_new - proj
            scores.append(torch.norm(residual, p=2, dim=1).mean().item())
    avg = np.mean(scores)
    print(f"--> Novelty Score: {avg:.4f}")
    if avg > 0.25:
        new_dim = int(np.ceil(64 * avg))
        new_dim = min(new_dim, 64) 
    else:
        new_dim = 4
    return new_dim, avg

def train_phase1(model, loader, test_loader, device):
    print("\n[Phase 1] Train Base Global Model (Task A)...")
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for ep in range(50):
        model.train()
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits, _ = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()
    return model

def train_phase2(loader, device):
    print("\n[Phase 2] Train Specialist (Task B)...")
    spec = Specialist().to(device)
    head = nn.Linear(64, 5).to(device)
    opt = torch.optim.AdamW(list(spec.parameters()) + list(head.parameters()), lr=1e-3)
    for ep in range(30):
        spec.train()
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device), (y-5).to(device)
            opt.zero_grad()
            logits = head(F.adaptive_avg_pool2d(spec(x), 1).flatten(1))
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()
    return spec

def train_phase3(model, loader_B, replay_loader, teacher, device):
    print("\n[Phase 3] Integration (Reinforced Hybrid)...")
    
    # Freeze logic
    for n, p in model.named_parameters(): p.requires_grad = False
    
    # Unfreeze new components
    for p in model.global_model.fusion.parameters(): p.requires_grad = True
    model.global_model.classifier.weight.requires_grad = True
    # FREEZE SIGMA (Advice #1)
    model.global_model.classifier.sigma.requires_grad = False
    
    opt = torch.optim.AdamW([
        {'params': model.global_model.fusion.parameters(), 'lr': 1e-3},
        {'params': model.global_model.classifier.parameters(), 'lr': 1e-3}
    ])
    
    replay_iter = iter(replay_loader)
    
    for ep in range(50):
        model.train()
        model.specialists[0].eval() 
        model.global_model.old_proj.eval()
        model.global_model.old_gate.eval()
        model.global_model.old_bottleneck.eval()
        model.specialists[1].eval()
        
        for x_B, y_B in tqdm(loader_B, leave=False):
            x_B, y_B = x_B.to(device), y_B.to(device)
            
            # Replay Batch
            try: x_A, y_A = next(replay_iter)
            except: 
                replay_iter = iter(replay_loader)
                x_A, y_A = next(replay_iter)
            x_A, y_A = x_A.to(device), y_A.to(device)
            
            opt.zero_grad()
            
            # 1. Task B Loss
            logits_B, _ = model(x_B)
            loss_B = F.cross_entropy(logits_B, y_B)
            
            # 2. Replay Loss (Anchoring)
            logits_A, _ = model(x_A)
            loss_replay = F.cross_entropy(logits_A, y_A)
            
            # 3. Distillation Loss (Advice #3)
            with torch.no_grad():
                logits_T, _ = teacher(x_A)
            # Distill only the old logits part
            loss_dist = F.kl_div(F.log_softmax(logits_A[:, :5], dim=1), 
                                 F.softmax(logits_T[:, :5], dim=1), reduction='batchmean')
            
            # Weighted Sum
            total_loss = loss_B + 1.0 * loss_replay + 2.0 * loss_dist
            
            total_loss.backward()
            
            # Structural Locking (Prevent overwrite of old weights)
            model.global_model.classifier.weight.grad[:5, :] = 0.0
            
            opt.step()
            
    return model

def evaluate(model, loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

def run():
    set_deterministic(42)
    device = get_device()
    os.makedirs("logs_reinforced", exist_ok=True)
    
    stats = ((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    trans = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])
    train_ds = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=trans)
    test_ds = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=trans)
    
    loader_A = DataLoader(get_class_subset(train_ds, range(5)), 128, shuffle=True)
    loader_B = DataLoader(get_class_subset(train_ds, range(5,10)), 128, shuffle=True)
    test_A = DataLoader(get_class_subset(test_ds, range(5)), 128)
    test_B = DataLoader(get_class_subset(test_ds, range(5,10)), 128)
    
    # Phase 1
    model = KFN(n_classes=5, n_specs=1).to(device)
    model = train_phase1(model, loader_A, test_A, device)
    acc_A_init = evaluate(model, test_A, device)
    
    # Teacher for distillation
    teacher = copy.deepcopy(model).eval()
    
    # Phase 2
    spec = train_phase2(loader_B, device)
    
    # Phase 3
    new_dim, novelty = compute_novelty(model, spec, test_B, device)
    model = expand_kfn(model, spec, new_dim, novelty)
    
    # Replay Buffer
    replay_loader = DataLoader(build_replay_buffer(train_ds, range(5), per_class=200), batch_size=32, shuffle=True)
    
    model = train_phase3(model, loader_B, replay_loader, teacher, device)
    
    acc_A = evaluate(model, test_A, device)
    acc_B = evaluate(model, test_B, device)
    
    print("\n" + "="*40)
    print("FINAL RESULTS (REINFORCED HYBRID)")
    print("="*40)
    print(f"Task A Initial: {acc_A_init:.2f}%")
    print(f"Task A Final:   {acc_A:.2f}%")
    print(f"Task B Final:   {acc_B:.2f}%")
    print(f"Retention:      {100*acc_A/acc_A_init:.1f}%")
    print("="*40)

if __name__ == "__main__":
    run()

100%|██████████| 170M/170M [00:05<00:00, 31.7MB/s] 



[Phase 1] Train Base Global Model (Task A)...



[Phase 2] Train Specialist (Task B)...



[Phase 3A] Novelty Analysis...


--> Novelty Score: 0.9989

[Phase 3] Integration (Reinforced Hybrid)...



FINAL RESULTS (REINFORCED HYBRID)
Task A Initial: 86.22%
Task A Final:   66.28%
Task B Final:   75.82%
Retention:      76.9%


In [9]:
from IPython.display import HTML

HTML("""
<audio id="player">
  <source src="https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3" type="audio/mp3">
</audio>

<script>
  var audio = document.getElementById("player");
  audio.play();
  
  // Stop after 5 seconds
  setTimeout(function() {
      audio.pause();
      audio.currentTime = 0;
  }, 5000);
</script>
""")
